## **Aim**
To implement a program that analyzes browser history data to identify frequently visited websites and suspicious URLs.

## **Algorithm**
**Step 1:** Import `sqlite3`, `collections.Counter`, `urllib.parse`, and `re` libraries.

**Step 2:** Create a simulated browser history database (SQLite) with tables for URLs and visits.

**Step 3:** Define suspicious patterns: known phishing domains, URL shorteners, suspicious TLDs, IP-based URLs, long subdomain chains.

**Step 4:** Extract all URLs from the history database.

**Step 5:** Parse URLs to extract domains and count visit frequencies.

**Step 6:** Check each URL against suspicious patterns and assign risk scores.

**Step 7:** Generate a report showing top visited sites and flagged suspicious URLs.

In [1]:
import sqlite3
import os
from collections import Counter
from urllib.parse import urlparse
import re

SUSPICIOUS_TLDS = {'.tk', '.ml', '.ga', '.cf', '.gq', '.xyz', '.top', '.club', '.work', '.date', '.bid', '.loan', '.racing', '.download', '.stream', '.science', '.party', '.review', '.cricket', '.win', '.accountant', '.faith', '.trade'}

URL_SHORTENERS = {'bit.ly', 'tinyurl.com', 'goo.gl', 't.co', 'ow.ly', 'is.gd', 'buff.ly', 'adf.ly', 'bit.do', 'short.io', 'cutt.ly', 'rb.gy'}

PHISHING_KEYWORDS = ['login', 'signin', 'verify', 'account', 'update', 'secure', 'bank', 'paypal', 'amazon', 'microsoft', 'apple', 'google', 'facebook', 'instagram', 'twitter', 'netflix', 'dropbox', 'onedrive', 'icloud']

def create_sample_history(db_path):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS urls (
            id INTEGER PRIMARY KEY,
            url TEXT NOT NULL,
            title TEXT,
            visit_count INTEGER DEFAULT 0,
            typed_count INTEGER DEFAULT 0,
            last_visit_time INTEGER
        )
    """)
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS visits (
            id INTEGER PRIMARY KEY,
            url_id INTEGER,
            visit_time INTEGER,
            FOREIGN KEY(url_id) REFERENCES urls(id)
        )
    """)
    
    # Sample history data
    sample_data = [
        ("https://www.google.com/search?q=python", "Google Search", 45),
        ("https://www.google.com/", "Google", 32),
        ("https://github.com/user/repo", "GitHub Repository", 18),
        ("https://stackoverflow.com/questions/123", "Stack Overflow", 15),
        ("https://www.youtube.com/watch?v=abc123", "YouTube Video", 22),
        ("https://mail.google.com/mail/u/0/", "Gmail", 28),
        ("https://drive.google.com/drive/my-drive", "Google Drive", 12),
        # Suspicious URLs
        ("http://paypal-login.verify-account.tk/", "PayPal Verify", 3),
        ("https://microsoft-security-update.xyz/login", "Microsoft Update", 5),
        ("http://192.168.1.100/admin/login.php", "Local Admin", 2),
        ("https://bit.ly/3xYzK9", "Short Link", 4),
        ("https://amazon.com.phishing-site.ml/order/confirm", "Amazon Order", 2),
        ("https://secure-bank-login.ga/verify", "Bank Verify", 1),
        ("https://docs.google.com/document/d/123/edit", "Google Docs", 10),
        ("https://www.linkedin.com/feed/", "LinkedIn", 8),
        ("https://tinyurl.com/malware-download", "TinyURL", 1),
    ]
    
    for url, title, count in sample_data:
        cursor.execute("INSERT INTO urls (url, title, visit_count) VALUES (?, ?, ?)", (url, title, count))
        url_id = cursor.lastrowid
        for _ in range(count):
            cursor.execute("INSERT INTO visits (url_id, visit_time) VALUES (?, ?)", (url_id, 1234567890))
    
    conn.commit()
    conn.close()

def analyze_url(url):
    parsed = urlparse(url)
    domain = parsed.netloc.lower()
    path = parsed.path.lower()
    full = url.lower()
    
    issues = []
    score = 0
    
    # IP address instead of domain
    if re.match(r'^\d+\.\d+\.\d+\.\d+', domain):
        issues.append("IP address used instead of domain")
        score += 30
    
    # Suspicious TLD
    for tld in SUSPICIOUS_TLDS:
        if domain.endswith(tld):
            issues.append(f"Suspicious TLD: {tld}")
            score += 25
            break
    
    # URL shortener
    if domain in URL_SHORTENERS:
        issues.append("URL shortener detected")
        score += 20
    
    # Phishing keywords in subdomain or path
    for kw in PHISHING_KEYWORDS:
        if kw in domain or kw in path:
            issues.append(f"Phishing keyword in URL: {kw}")
            score += 15
            break
    
    # Long subdomain chain (many dots)
    if domain.count('.') >= 3:
        issues.append(f"Long subdomain chain ({domain.count('.')} dots)")
        score += 10
    
    # Brand name in subdomain (not main domain)
    brands = ['paypal', 'microsoft', 'apple', 'google', 'amazon', 'facebook', 'netflix', 'bank', 'chase', 'wellsfargo']
    for brand in brands:
        if brand in domain and not domain.startswith(brand + '.') and not domain == brand + '.com':
            issues.append(f"Brand name in subdomain: {brand}")
            score += 20
            break
    
    # HTTP instead of HTTPS
    if parsed.scheme == 'http':
        issues.append("Insecure HTTP protocol")
        score += 5
    
    if score >= 40:
        risk = "HIGH"
    elif score >= 20:
        risk = "MEDIUM"
    elif score > 0:
        risk = "LOW"
    else:
        risk = "SAFE"
    
    return {"domain": domain, "risk": risk, "score": score, "issues": issues}

def main():
    db_path = "browser_history.db"
    
    if os.path.exists(db_path):
        os.remove(db_path)
    
    create_sample_history(db_path)
    
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("SELECT url, title, visit_count FROM urls ORDER BY visit_count DESC")
    rows = cursor.fetchall()
    conn.close()
    
    print(f"{'URL':<55} {'Visits':<8} {'Risk':<8} Issues")
    print("-" * 100)
    
    domain_counter = Counter()
    suspicious_count = 0
    
    for url, title, count in rows:
        analysis = analyze_url(url)
        domain_counter[analysis['domain']] += count
        
        issues_str = "; ".join(analysis['issues']) if analysis['issues'] else "None"
        risk_marker = " >>>" if analysis['risk'] in ('HIGH', 'MEDIUM') else ""
        
        display_url = url[:52] + "..." if len(url) > 55 else url
        print(f"{display_url:<55} {count:<8} {analysis['risk']:<8} {issues_str}{risk_marker}")
        
        if analysis['risk'] in ('HIGH', 'MEDIUM'):
            suspicious_count += 1
    
    print(f"\n{'='*50}")
    print(f"TOP 10 MOST VISITED DOMAINS")
    print(f"{'='*50}")
    for domain, visits in domain_counter.most_common(10):
        print(f"  {domain:<40} {visits} visits")
    
    print(f"\nTotal URLs analyzed: {len(rows)}")
    print(f"Suspicious URLs flagged: {suspicious_count}")

if __name__ == "__main__":
    main()

URL                                                     Visits   Risk     Issues
----------------------------------------------------------------------------------------------------
https://www.google.com/search?q=python                  45       MEDIUM   Phishing keyword in URL: google; Brand name in subdomain: google >>>
https://www.google.com/                                 32       MEDIUM   Phishing keyword in URL: google; Brand name in subdomain: google >>>
https://mail.google.com/mail/u/0/                       28       MEDIUM   Phishing keyword in URL: google; Brand name in subdomain: google >>>
https://www.youtube.com/watch?v=abc123                  22       SAFE     None
https://github.com/user/repo                            18       SAFE     None
https://stackoverflow.com/questions/123                 15       SAFE     None
https://drive.google.com/drive/my-drive                 12       MEDIUM   Phishing keyword in URL: google; Brand name in subdomain: google >>>
https://d

## **Result**
This the program successfully analyzes browser history data and identifies frequently visited websites and suspicious URLs.